In [1]:
import polars as pl
import pathlib

In [2]:
path = r'D:\Trading\2026\eurusd_h1_sep_2026.csv'

In [3]:
CCYS = ['jpy','chf','cad','aud','nzd','eur','gbp']
INVERSE_CCYS = ['chf','jpy','cad']
def get_read_path(ccy: str):
    if ccy in INVERSE_CCYS:
        pair = f'usd{ccy}'
    else:
        pair = f'{ccy}usd'
    return pathlib.Path(r'D:\Trading\2026\\',f"{pair}_h1_sep_2026.csv")
def get_write_path(ccy: str):
    if ccy in INVERSE_CCYS:
        pair = f'usd{ccy}'
    else:
        pair = f'{ccy}usd'
    return pathlib.Path(r'D:\Trading\2026\\',f"{pair}_d1_sep_2026_08.csv")

In [4]:
for ccy in CCYS:
    df = pl.read_csv(get_read_path(ccy))

    df = df.with_columns(
        pl.from_epoch("timestamp", time_unit="ms")
          .dt.cast_time_unit("ns")
          .dt.replace_time_zone("UTC")
    )
    df = df.with_columns(
        (pl.col("timestamp") - pl.duration(hours=8)).alias("shifted_ts")
    )



    daily = (
        df.with_columns(pl.col("shifted_ts").dt.date().alias("session_date"))
        .group_by("session_date", maintain_order=True)
        .agg([
            pl.col("open").first().alias("open"),
            pl.col("high").max().alias("high"),
            pl.col("low").min().alias("low"),
            pl.col("close").last().alias("close"),
        ])
        .sort("session_date")
        .with_columns(
            pl.col("session_date")
            .cast(pl.Datetime("ms"))
            .dt.epoch(time_unit="ms")
            .alias("timestamp")
        )
        .select(["timestamp", "open", "high", "low", "close"])
    )

    daily.write_csv(get_write_path(ccy))


In [25]:
df = pl.read_csv(path)


In [7]:
daily

session_date,session_open_time,session_close_time,open,high,low,close
date,"datetime[ns, UTC]","datetime[ns, UTC]",f64,f64,f64,f64
2003-05-04,2003-05-04 19:00:00 UTC,2003-05-05 07:00:00 UTC,1.12305,1.12338,1.12014,1.12143
2003-05-05,2003-05-05 08:00:00 UTC,2003-05-06 07:00:00 UTC,1.12129,1.13009,1.12042,1.1293
2003-05-06,2003-05-06 08:00:00 UTC,2003-05-07 07:00:00 UTC,1.12956,1.14506,1.12933,1.14068
2003-05-07,2003-05-07 08:00:00 UTC,2003-05-08 07:00:00 UTC,1.14061,1.14302,1.13006,1.13306
2003-05-08,2003-05-08 08:00:00 UTC,2003-05-09 07:00:00 UTC,1.1326,1.15366,1.13075,1.14978
…,…,…,…,…,…,…
2026-08-27,2026-08-27 08:00:00 UTC,2026-08-28 07:00:00 UTC,1.16548,1.16598,1.16364,1.16464
2026-08-28,2026-08-28 08:00:00 UTC,2026-08-29 07:00:00 UTC,1.16464,1.16589,1.15777,1.15828
2026-08-29,2026-08-29 08:00:00 UTC,2026-08-30 07:00:00 UTC,1.15828,1.15828,1.15828,1.15828
